# 70장 · SQLite 문서 상태와 대화 턴 저장

파일 접수와 검색 준비 완료는 다른 상태입니다. 처리 상태를 저장하고, 답변이 끝나면 한 턴의 본문과 출처를 함께 기록합니다.

SQL 데이터베이스에서 문서 접수, 본문 처리 완료, 대화 저장을 구분합니다. 파일은 접수됐지만 아직 질문에 사용할 수 없는 상태를 조회 결과로 확인합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 문서 접수와 검색 가능 상태 전이

아래는 70장의 소유자·문서·상태 셋을 남긴 SQLite 실습입니다. 실제 PostgreSQL 버전은 UUID와 복합 외래키로 문서·대화·출처를 연결합니다. 여기서는 업로드 본문 한 건의 변화를 관찰합니다.


### received와 ready를 가르는 조건

<p>접수 직후 doc-1의 body는 NULL이고 status는 received입니다. 본문을 디코딩해 저장하면서 같은 행의 상태를 ready로 바꿉니다. 검색할 때는 ready인 행만 선택하므로 접수만 끝난 문서는 답변 근거에 들어가지 않습니다.</p>


### UTF-8 바이트를 문서 본문으로 읽기

<p>문자열을 encode("utf-8") 하면 바이트가 됩니다. 셀에서는 같은 인코딩으로 decode한 텍스트를 body에 저장합니다. bytes를 그대로 저장할 때와 읽어서 답변에 사용할 문자열을 저장할 때의 차이를 확인하세요.</p>


### 셀 01 · 문서 접수 상태 저장

과제 안내를 alice 문서로 접수합니다. 텍스트를 저장하기 전에는 ready로 표시하지 않습니다.


In [1]:
import sqlite3, json
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE docs(id TEXT PRIMARY KEY, owner TEXT, status TEXT, body TEXT)")
db.execute("CREATE TABLE turns(conversation TEXT, owner TEXT, result TEXT)")
db.execute("INSERT INTO docs VALUES (?, ?, 'received', NULL)", ("doc-1", "alice"))
print(db.execute("SELECT status FROM docs WHERE id=? AND owner=?", ("doc-1", "alice")).fetchone()[0])

received


received는 요청을 접수했다는 뜻입니다. 원문 처리나 출처 준비 성공을 보장하지 않습니다.


### 셀 02 · UTF-8 디코딩과 상태 갱신

기존 상태가 received일 때만 한 번 전환합니다. 자료가 준비된 뒤에만 검색 후보에 넣으세요.

같은 셀을 두 번 실행해 변경 행 수를 비교하세요. 처음에는 received 상태의 문서를 찾아 바꾸고, 두 번째에는 이미 ready라서 UPDATE에 맞는 행이 없습니다. 조회한 최종 상태는 둘 다 ready입니다.


In [2]:
raw = "과제는 학습 관리 페이지의 과제함에 제출합니다.".encode("utf-8")
with db:
    change = db.execute("UPDATE docs SET body=?, status='ready' WHERE id=? AND owner=? AND status='received'",
               (raw.decode("utf-8"), "doc-1", "alice"))
print("변경한 행:", change.rowcount)
print(db.execute("SELECT status, body FROM docs WHERE id='doc-1'").fetchone())

변경한 행: 1
('ready', '과제는 학습 관리 페이지의 과제함에 제출합니다.')


이 UPDATE는 디코딩한 body와 ready를 같은 행에 기록합니다. 디코딩이 실패하면 SQL을 호출하기 전에 예외가 납니다. 같은 셀을 다시 실행했을 때 변경 수가 0이라면 새로 처리한 행이 없었다는 뜻입니다.


### UPDATE의 세 조건

<p>doc-1이라는 ID, alice라는 소유자, received라는 이전 상태가 모두 맞아야 UPDATE가 한 행을 바꿉니다. 첫 갱신 직후 같은 UPDATE를 다시 실행한다면 상태가 이미 ready이므로 변경한 행 수는 0이 됩니다.</p>


## 답변·출처 저장과 소유자별 조회

답변 문장만 저장하면 같은 대화를 다시 열 때 어떤 문서와 쪽수를 썼는지 복원하기 어렵습니다. 답변과 출처 목록을 JSON 하나로 저장한 뒤 다시 읽습니다. 수업용 doc-1 식별자는 업로드한 문서를 가리킵니다.


### 셀 03 · 저장된 턴에서 출처 조회

alice의 ready 문서에서 본문을 읽어 답변과 출처를 구성합니다. generation_calls의 1은 저장할 JSON에 직접 넣은 예시 값이며 이 셀에서 모델을 호출하지 않습니다.


In [3]:
body = db.execute("SELECT body FROM docs WHERE owner=? AND id=? AND status='ready'",
                  ("alice", "doc-1")).fetchone()[0]
source = {"id": "doc-1:p1", "page": 1, "source": "제출 안내", "text": body}
a1 = {"status": "answered", "answer": body + " [doc-1:p1]", "sources": [source], "generation_calls": 1}
with db:
    db.execute("INSERT INTO turns VALUES (?,?,?)", ("chat-a", "alice", json.dumps(a1, ensure_ascii=False)))
saved = db.execute("SELECT result FROM turns WHERE owner=? AND conversation=?", ("alice", "chat-a")).fetchone()
print(json.loads(saved[0])["sources"][0]["text"])

과제는 학습 관리 페이지의 과제함에 제출합니다.


다시 조회한 행에도 원문이 남습니다. 이 코드는 저장까지 성공했습니다. 답변 생성 뒤 저장에 실패했다면 응답이 생성된 사실과 이력이 남았는지를 별도로 표시해야 합니다.


### 한 대화 턴에 답변과 출처를 함께 저장하기

<p>먼저 답변에 쓴 원문과 페이지를 source에 넣습니다. 둘을 포함한 결과를 JSON 문자열 하나로 저장하므로, 다시 불러올 때 어떤 본문으로 답했는지도 복원할 수 있습니다.</p>


### 소유자와 대화 이름을 함께 조회하기

<p>alice의 chat-a에는 결과가 한 건 있습니다. 대화 이름을 그대로 두고 소유자만 bob으로 바꾸면 두 조건을 모두 만족하는 행이 없어 fetchone()이 None을 돌려줍니다.</p>


### 과제 04 · 소유자 조건을 포함한 대화 조회

조회 함수에 소유자와 대화 이름을 전달하세요. alice가 읽은 답변의 출처 ID와, 같은 chat-a를 bob으로 읽은 결과를 순서대로 출력하세요.


In [4]:
def load_turn(owner, conversation):
    # 소유자와 대화 이름으로 조회하고, 찾은 JSON만 복원하세요.
    pass

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def load_turn(owner, conversation):
    row = db.execute(&quot;SELECT result FROM turns WHERE owner=? AND conversation=?&quot;,
                     (owner, conversation)).fetchone()
    return json.loads(row[0]) if row is not None else None

print(load_turn(&quot;alice&quot;, &quot;chat-a&quot;)[&quot;sources&quot;][0][&quot;id&quot;])
print(load_turn(&quot;bob&quot;, &quot;chat-a&quot;))</code></pre>
<p>실행 결과</p><pre><code>doc-1:p1
None</code></pre>
<p>두 번의 조회에서 바뀐 값은 소유자뿐입니다. 항상 None을 반환하는 잘못된 함수도 bob 한 건만 시험하면 통과합니다. alice가 원래 저장한 출처까지 받는지 함께 확인합니다.</p>
</details>


## 돌아보기

ready인데 원문이 NULL이라면 문서 처리 단계의 상태와 저장값이 맞지 않습니다. 문서는 정상인데 대화를 다시 열었을 때 출처가 없다면 turns.result에 저장한 JSON부터 확인합니다.
